# Setup

---



In [ ]:
# Standard library
import json
import time
from datetime import datetime
from pathlib import Path
from typing import Optional

# Third-party libraries
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd

# scikit-learn
from sklearn.metrics import mean_absolute_error

# Settings
---

In [ ]:
# -----------------------------------------------------------------------------
# Repository root
# -----------------------------------------------------------------------------
REPO_ROOT = Path.cwd().parent.parent

# -----------------------------------------------------------------------------
# Global configuration
# -----------------------------------------------------------------------------
TARGET_TZ = "Europe/Berlin"

# -----------------------------------------------------------------------------
# SQRA
# -----------------------------------------------------------------------------
IMPORT_P1 = REPO_ROOT / "results" / "lear_op_results" / "dwd" / "d56" / "c1" / "exaa" / "forecast.csv"
IMPORT_P2 = REPO_ROOT / "results" / "lear_op_results" / "dwd" / "d56" / "c5" / "fundamental" / "forecast.csv"
IMPORT_P3 = REPO_ROOT / "results" / "lear_op_results" / "era5" / "d364" / "c5" / "fundamental" / "forecast.csv"
IMPORT_PATHS_SQRA = [IMPORT_P1, IMPORT_P2, IMPORT_P3]

QUANTILES_SQRA = [0.10, 0.25, 0.50, 0.75, 0.90]

TEST_START_SQRA = pd.Timestamp("2025-12-01", tz=TARGET_TZ)
TEST_END_SQRA = pd.Timestamp("2026-02-28 23:45", tz=TARGET_TZ)
TRAIN_DAYS_ROLLING_SQRA = 60

EXPERIMENT_NAME_SQRA = "sqra_dwd_exaa_enriched_d60"

EXPORT_BASE_SQRA = REPO_ROOT / "results" / "sqra_results" / "dwd_exaa_enriched"

# SQRA
---

## Model

### Imports

In [ ]:
%pip install remodels

In [ ]:
from remodels.qra import SQRA

### Functions for Rolling Forecast Loop

In [ ]:
def fit_sqra_for_mtu_and_quantile(
    df: pd.DataFrame,
    train_mask: np.ndarray,
    mtu: int,
    quantile: float,
    feature_cols: list[str],
) -> SQRA:
    """
    Fit SQRA model for one MTU and one quantile.

    Parameters
    ----------
    df : pd.DataFrame
        Full panel DataFrame with columns for features, 'y_true', and 'mtu'.
    train_mask : np.ndarray
        Boolean mask selecting the training window rows from df.
    mtu : int
        MTU index (0..95) to fit the model for.
    quantile : float
        Target quantile (e.g. 0.1, 0.5, 0.9).
    feature_cols : list[str]
        List of feature column names to use as regressors.

    Returns
    -------
    SQRA or None
        Fitted SQRA model, or None if fitting failed or no valid data exists.
    """
    idx = train_mask & (df["mtu"] == mtu)

    if idx.sum() == 0:
        return None

    X_train = df.loc[idx, feature_cols]
    y_train = df.loc[idx, "y_true"]

    valid = X_train.notna().all(axis=1) & y_train.notna()

    if valid.sum() == 0:
        return None

    X_train = X_train.loc[valid].to_numpy()
    y_train = y_train.loc[valid].to_numpy()

    model = SQRA(quantile=quantile, fit_intercept=True)

    try:
        model.fit(X_train, y_train)
    except Exception:
        return None

    return model

In [ ]:
def predict_sqra_for_mtu(
    df: pd.DataFrame,
    test_mask: np.ndarray,
    mtu: int,
    model: SQRA,
    feature_cols: list[str],
) -> pd.Series:
    """
    Generate SQRA predictions for a single MTU on the forecast day.

    Parameters
    ----------
    df : pd.DataFrame
        Full panel DataFrame with columns for features and 'mtu'.
    test_mask : np.ndarray
        Boolean mask selecting the forecast day rows from df.
    mtu : int
        MTU index (0..95) to predict for.
    model : SQRA or None
        Fitted SQRA model as returned by fit_sqra_for_mtu_and_quantile().
        If None, NaN predictions are returned.
    feature_cols : list[str]
        List of feature column names to use as regressors.

    Returns
    -------
    pd.Series
        Quantile predictions indexed by timestamp. NaN where prediction
        is not possible.
    """
    # Select observations belonging to the forecast day and the given MTU
    idx = test_mask & (df["mtu"] == mtu)

    # If no observations exist for this MTU, return an empty Series
    if not idx.any():
        return pd.Series(dtype=float)

    # Extract the correct timestamp index for the forecast horizon
    index = df.loc[idx].index

    # If the SQRA model could not be estimated, return NaN predictions
    if model is None:
        return pd.Series(index=index, dtype=float)

    # Build the test feature matrix
    X_test = df.loc[idx, feature_cols]

    # Identify rows without missing feature values
    valid = X_test.notna().all(axis=1)

    # If all rows contain missing values, return NaN predictions
    if not valid.any():
        return pd.Series(index=index, dtype=float)

    # Prepare the prediction Series with the full forecast index
    preds = pd.Series(index=index, dtype=float)

    # Predict only for rows with valid feature values
    preds.loc[valid] = model.predict(X_test.loc[valid].to_numpy())

    return preds

In [ ]:
def rolling_sqra_forecast_mtu(
    df: pd.DataFrame,
    forecast_days: list[pd.Timestamp],
    train_days: int,
    quantiles: list[float],
    feature_cols: list[str],
) -> tuple[pd.DataFrame, pd.DataFrame]:
    """
    Generate rolling SQRA quantile forecasts for all MTUs and forecast days.

    Parameters
    ----------
    df : pd.DataFrame
        Full panel DataFrame with columns for features, 'y_true', and 'mtu',
        indexed by timezone-aware timestamps (Europe/Berlin).
    forecast_days : list[pd.Timestamp]
        Ordered list of days to forecast.
    train_days : int
        Number of calendar days in the rolling training window.
    quantiles : list[float]
        List of quantile levels to forecast (e.g. [0.1, 0.25, 0.5, 0.75, 0.9]).
    feature_cols : list[str]
        List of feature column names to use as regressors.

    Returns
    -------
    forecast_df : pd.DataFrame
        Quantile forecasts and y_true per MTU-level timestamp.
        Columns: q{tau:.3f} for each quantile, plus 'y_true'.
    runtime_df : pd.DataFrame
        Per-day computation times with columns 'forecast_day' and
        'computation_time_seconds'.
    """
    all_days = []
    runtime_records = []

    for day in forecast_days:
        print(day)

        # Start runtime measurement for the current forecast day
        start_time = time.perf_counter()

        # Define rolling training and forecast windows
        train_start = day - pd.Timedelta(days=train_days)
        train_end = day - pd.Timedelta(minutes=15)
        test_end = day + pd.Timedelta(days=1)

        train_mask = (df.index >= train_start) & (df.index <= train_end)
        test_mask = (df.index >= day) & (df.index < test_end)

        # Initialize result container for the current forecast day
        day_index = df.loc[test_mask].index
        day_df = pd.DataFrame(index=day_index)

        for tau in quantiles:
            preds_tau = []

            for mtu in range(96):
                # Fit the SQRA model for one MTU and one quantile
                model = fit_sqra_for_mtu_and_quantile(
                    df=df,
                    train_mask=train_mask,
                    mtu=mtu,
                    quantile=tau,
                    feature_cols=feature_cols,
                )

                # Predict the quantile for the current MTU on the forecast day
                preds_mtu = predict_sqra_for_mtu(
                    df=df,
                    test_mask=test_mask,
                    mtu=mtu,
                    model=model,
                    feature_cols=feature_cols,
                )

                preds_tau.append(preds_mtu)

            # Combine MTU-level predictions into a full-day quantile forecast
            day_df[f"q{tau:.3f}"] = pd.concat(preds_tau).sort_index()

        # Add realized values for later evaluation
        day_df["y_true"] = df.loc[test_mask, "y_true"]

        all_days.append(day_df)

        # Stop runtime measurement and store it for the current forecast day
        runtime_seconds = time.perf_counter() - start_time
        runtime_records.append({
            "forecast_day": day,
            "computation_time_seconds": runtime_seconds,
        })

    if all_days:
        forecast_df = pd.concat(all_days).sort_index()
    else:
        forecast_df = pd.DataFrame()

    runtime_df = pd.DataFrame(runtime_records)

    return forecast_df, runtime_df

In [ ]:
# Post-processing: enforce monotonic quantile forecasts
def sort_quantiles(
    df: pd.DataFrame,
    quantile_cols: list[str],
) -> pd.DataFrame:
    """
    Enforce monotonicity of quantile forecasts per timestamp.

    Quantile regression models may occasionally produce crossing
    quantiles (e.g., q0.9 < q0.8). This function fixes such violations
    by sorting the quantile values row-wise.

    Parameters
    ----------
    df : pd.DataFrame
        Forecast DataFrame containing quantile columns.
    quantile_cols : list[str]
        Ordered list of quantile column names (e.g. ['q0.100', 'q0.500', 'q0.900']).

    Returns
    -------
    pd.DataFrame
        Copy of df with quantile columns sorted row-wise to enforce monotonicity.
    """
    df_sorted = df.copy()

    # Ensure the requested quantile columns exist
    missing_cols = [c for c in quantile_cols if c not in df.columns]
    if missing_cols:
        raise ValueError(f"Missing quantile columns: {missing_cols}")

    # Sort quantile values row-wise to enforce monotonicity
    sorted_values = np.sort(df_sorted[quantile_cols].to_numpy(), axis=1)

    # Write sorted values back to the DataFrame
    df_sorted[quantile_cols] = sorted_values

    return df_sorted

### Evaluation: Plot and Metrics

In [ ]:
def plot_prob_forecast(
    df_forecast: pd.DataFrame,
    quantile_cols: list[str],
    start_date: str,
    end_date: str,
    y_true_col: str = "y_true",
    title: str = "",
    model_info: Optional[str] = None,
):
    """
    Plot probabilistic forecast with prediction intervals and median.

    quantile_cols must be sorted in ascending quantile order.

    Parameters
    ----------
    df_forecast : pd.DataFrame
        Forecast DataFrame with quantile columns and y_true.
    quantile_cols : list[str]
        Ordered list of quantile column names in ascending order
        (e.g. ['q0.100', 'q0.250', 'q0.500', 'q0.750', 'q0.900']).
    start_date : str
        Start of the plot window (date string, e.g. '2025-12-01').
    end_date : str
        End of the plot window (date string, e.g. '2026-02-28').
    y_true_col : str
        Column name for realized values. Defaults to 'y_true'.
    title : str
        Plot title. Defaults to ''.
    model_info : str, optional
        Model description shown as a text box in the top-left corner.

    Returns
    -------
    None
    """
    df = df_forecast.loc[start_date:end_date]

    if df.empty:
        raise ValueError("Selected time window is empty.")

    # Identify intervals dynamically
    n            = len(quantile_cols)
    q_outer_low  = quantile_cols[0]
    q_outer_high = quantile_cols[-1]
    q_median     = quantile_cols[n // 2]

    if n >= 4:
        q_inner_low  = quantile_cols[1]
        q_inner_high = quantile_cols[-2]
    else:
        q_inner_low = q_inner_high = None

    plt.figure(figsize=(18, 6))

    # Outer PI
    if q_outer_low in df.columns and q_outer_high in df.columns:
        plt.fill_between(
            df.index, df[q_outer_low], df[q_outer_high],
            alpha=0.2,
            label="Outer PI",
        )

    # Inner PI
    if q_inner_low and q_inner_high:
        if q_inner_low in df.columns and q_inner_high in df.columns:
            plt.fill_between(
                df.index, df[q_inner_low], df[q_inner_high],
                alpha=0.4,
                label="Inner PI",
            )

    # Median
    if q_median in df.columns:
        plt.plot(
            df.index, df[q_median],
            linewidth=2,
            label="Median forecast",
        )

    # Actual
    if y_true_col in df.columns:
        plt.plot(
            df.index, df[y_true_col],
            linewidth=2,
            label="Actual",
            color="#3b5b73",
        )

    plt.title(title)
    plt.xlabel("Date")
    plt.ylabel("Electricity Price [EUR/MWh]")
    plt.grid(alpha=0.3)
    plt.legend(loc="upper right")

    if model_info is not None:
        plt.gca().text(
            0.01, 0.99,
            model_info,
            transform=plt.gca().transAxes,
            fontsize=10,
            verticalalignment="top",
            bbox=dict(boxstyle="round", facecolor="white", alpha=0.8),
        )

    plt.tight_layout()
    plt.show()

In [ ]:
def mae_median(
    df: pd.DataFrame,
    q_median: float = 0.5,
    y_true_col: str = "y_true",
):
    """
    Compute MAE between realized values and the median quantile forecast.

    Parameters
    ----------
    df : pd.DataFrame
        Forecast DataFrame with quantile columns and y_true.
    q_median : float
        Quantile level of the median forecast. Defaults to 0.5.
    y_true_col : str
        Column name for realized values. Defaults to 'y_true'.

    Returns
    -------
    float
        Mean absolute error of the median forecast.
    """
    col_median = f"q{q_median:.3f}"
    return mean_absolute_error(df[y_true_col], df[col_median])


def empirical_coverage(
    df: pd.DataFrame,
    lower_q: float,
    upper_q: float,
    y_true_col: str = "y_true",
):
    """
    Compute empirical coverage of a prediction interval.

    Parameters
    ----------
    df : pd.DataFrame
        Forecast DataFrame with quantile columns and y_true.
    lower_q : float
        Lower quantile level (e.g. 0.1).
    upper_q : float
        Upper quantile level (e.g. 0.9).
    y_true_col : str
        Column name for realized values. Defaults to 'y_true'.

    Returns
    -------
    float
        Share of observations falling within the prediction interval.
    """
    col_lower = f"q{lower_q:.3f}"
    col_upper = f"q{upper_q:.3f}"

    inside = (
        (df[y_true_col] >= df[col_lower]) &
        (df[y_true_col] <= df[col_upper])
    )

    return inside.mean()


def pinball_score(y_true, y_q, q):
    """
    Compute the pinball (quantile) loss for a single quantile.

    Parameters
    ----------
    y_true : array-like
        Realized values.
    y_q : array-like
        Quantile forecast values.
    q : float
        Quantile level (e.g. 0.1, 0.5, 0.9).

    Returns
    -------
    float
        Mean pinball loss.
    """
    diff = y_true - y_q

    return np.mean(np.maximum(q * diff, (q - 1) * diff))


def aggregate_pinball_score(
    df: pd.DataFrame,
    quantiles: list,
    y_true_col: str = "y_true",
):
    """
    Compute the average pinball score (APS) across all quantile levels.

    Parameters
    ----------
    df : pd.DataFrame
        Forecast DataFrame with quantile columns and y_true.
    quantiles : list
        List of quantile levels (e.g. [0.1, 0.25, 0.5, 0.75, 0.9]).
    y_true_col : str
        Column name for realized values. Defaults to 'y_true'.

    Returns
    -------
    scores : dict
        Per-quantile pinball scores keyed by quantile level.
    aps : float
        Average pinball score across all quantile levels.
    """
    scores = {}
    y_true = df[y_true_col]

    for q in quantiles:
        col = f"q{q:.3f}"
        scores[q] = pinball_score(y_true, df[col], q)

    aps = np.mean(list(scores.values()))
    return scores, aps

In [ ]:
def evaluate_probabilistic_forecasts(
    df: pd.DataFrame,
    start_date: str,
    end_date: str,
    quantiles: list,
    y_true_col: str = "y_true",
):
    """
    Evaluate probabilistic forecasts per calendar day over a given window.

    Computes MAE of the median forecast, empirical coverage of inner and
    outer prediction intervals, and the average pinball score (APS) for
    each day in the evaluation window. Returns a summary row appended at
    the end.

    Parameters
    ----------
    df : pd.DataFrame
        Forecast DataFrame with quantile columns and y_true, indexed by
        timezone-aware timestamps (Europe/Berlin).
    start_date : str
        Start of the evaluation window (date string, e.g. '2025-12-01').
    end_date : str
        End of the evaluation window (date string, e.g. '2026-02-28').
    quantiles : list
        List of quantile levels. Must contain 0.5 and at least 3 values
        symmetric around 0.5.
    y_true_col : str
        Column name for realized values. Defaults to 'y_true'.

    Returns
    -------
    pd.DataFrame
        Per-day evaluation metrics with a final summary row ('Mean over all days').
        Columns: 'Target Day', 'MAE (median)', coverage columns, 'APS'.
    """
    quantiles = sorted(quantiles)

    # --- sanity checks ---
    if 0.5 not in quantiles:
        raise ValueError("Quantiles must contain 0.5 for MAE evaluation.")

    if len(quantiles) < 3:
        raise ValueError("Need at least 3 quantiles to form prediction intervals.")

    # --- derive intervals automatically ---
    median_q = 0.5

    mid_idx = quantiles.index(0.5)
    if mid_idx == 0 or mid_idx == len(quantiles) - 1:
        raise ValueError("Quantiles must be symmetric around 0.5.")

    q_inner_low  = quantiles[mid_idx - 1]
    q_inner_high = quantiles[mid_idx + 1]
    q_outer_low  = quantiles[0]
    q_outer_high = quantiles[-1]

    tz = df.index.tz
    date_range = pd.date_range(
        start=start_date,
        end=end_date,
        freq="D",
        tz=tz,
    )

    results = []

    for day in date_range:
        day_end = day + pd.Timedelta(days=1)
        df_day = df.loc[(df.index >= day) & (df.index < day_end)]

        if df_day.empty:
            continue

        mae       = mae_median(df_day, q_median=median_q, y_true_col=y_true_col)
        cov_inner = empirical_coverage(df_day, q_inner_low, q_inner_high, y_true_col)
        cov_outer = empirical_coverage(df_day, q_outer_low, q_outer_high, y_true_col)
        _, aps    = aggregate_pinball_score(df_day, quantiles, y_true_col)

        results.append({
            "Target Day":                              str(day.date()),
            "MAE (median)":                            mae,
            f"Coverage {q_inner_low}-{q_inner_high}": cov_inner,
            f"Coverage {q_outer_low}-{q_outer_high}": cov_outer,
            "APS":                                     aps,
        })

    df_res  = pd.DataFrame(results)
    summary = df_res.mean(numeric_only=True)
    summary["Target Day"] = "Mean over all days"

    return pd.concat([df_res, pd.DataFrame([summary])], ignore_index=True)

## **Execution**

#### Load Point Forecast Panel

In [ ]:
def load_forecast(path: Path) -> pd.DataFrame:
    """
    Load a point forecast CSV and return a timezone-aware DataFrame.

    Parameters
    ----------
    path : Path
        Path to the forecast CSV file.

    Returns
    -------
    pd.DataFrame
        DataFrame with timezone-aware index (Europe/Berlin).
    """
    df = pd.read_csv(path, index_col=0)
    df.index = pd.to_datetime(df.index, utc=True).tz_convert("Europe/Berlin")
    return df

forecasts = [load_forecast(p) for p in IMPORT_PATHS_SQRA]
SQRA_FEATURE_COLS = [f"prediction_p{i+1}" for i in range(len(forecasts))]

In [ ]:
df_qra = pd.DataFrame(index=forecasts[0].index)
for i, fc in enumerate(forecasts):
    df_qra[f"prediction_p{i+1}"] = fc["y_pred"]
df_qra["y_true"] = forecasts[0]["y_true"]
df_qra["mtu"]    = df_qra.index.hour * 4 + df_qra.index.minute // 15

# Sanity check
print(f"Shape      : {df_qra.shape}")
print(f"Date range : {df_qra.index.min().date()} → {df_qra.index.max().date()}")
print(f"NaNs       :\n{df_qra.isna().sum()}")

In [ ]:
QUANTILE_COLS = [f"q{tau:.3f}" for tau in QUANTILES_SQRA]

In [ ]:
df_qra.head()

In [ ]:
# Basic sanity checks for the SQRA input data
print("Shape:", df_qra.shape)
print("Columns:", df_qra.columns.tolist())

## NaN check
nan_counts = df_qra[SQRA_FEATURE_COLS + ["y_true", "mtu"]].isna().sum()
print("\nNaNs per column:\n", nan_counts)

## MTU range check
print("\nMTU min/max:", df_qra["mtu"].min(), df_qra["mtu"].max())
print("Unique MTUs:", df_qra["mtu"].nunique())

## Quick index check
print("\nIndex tz:", df_qra.index.tz)
print("Index freq (inferred):", pd.infer_freq(df_qra.index[:2000]))

#### Define Forecast Window

In [ ]:
forecast_days_sqra = pd.date_range(
    start=TEST_START_SQRA.normalize(),
    end=TEST_END_SQRA.normalize(),
    freq="D",
    tz=TARGET_TZ,
)

print(f"Number of forecast days : {len(forecast_days_sqra)}")
print(f"First day               : {forecast_days_sqra[0].date()}")
print(f"Last day                : {forecast_days_sqra[-1].date()}")

#### Run Rolling Forecast Loop

In [ ]:
res_sqra, runtime_sqra = rolling_sqra_forecast_mtu(
    df=df_qra,
    forecast_days=forecast_days_sqra,
    train_days=TRAIN_DAYS_ROLLING_SQRA,
    quantiles=QUANTILES_SQRA,
    feature_cols=SQRA_FEATURE_COLS,
)

In [ ]:
# Print timestamps of NaN rows
nan_dates = res_sqra[res_sqra[QUANTILE_COLS].isna().any(axis=1)].index
print(f"Number of NaN timestamps: {len(nan_dates)}")
print(nan_dates)

#### Quantile Sorting

In [ ]:
# Sanity Check
print(f"Shape      : {res_sqra.shape}")
print(f"Date range : {res_sqra.index.min().date()} → {res_sqra.index.max().date()}")
print(f"NaNs       :\n{res_sqra.isna().sum()}")

crossing_count = (
    res_sqra[QUANTILE_COLS]
    .diff(axis=1)
    .iloc[:, 1:]
    .lt(0)
    .any(axis=1)
    .sum()
)
print(f"\nQuantile crossings before sorting: {crossing_count}")

In [ ]:
# Enforce monotonicity of quantile forecasts
res_sqra_sorted = sort_quantiles(
    df=res_sqra,
    quantile_cols=QUANTILE_COLS,
)

# Verify
crossing_count_after = (
    res_sqra_sorted[QUANTILE_COLS]
    .diff(axis=1)
    .iloc[:, 1:]
    .lt(0)
    .any(axis=1)
    .sum()
)
print(f"Quantile crossings after sorting: {crossing_count_after}")

#### Save Experiments Output

In [ ]:
EXPORT_BASE_SQRA.mkdir(parents=True, exist_ok=True)

res_sqra_sorted.to_csv(EXPORT_BASE_SQRA / "forecast.csv", index=True)
runtime_sqra.to_csv(EXPORT_BASE_SQRA / "runtime.csv", index=False)

meta = {
    "experiment_name": EXPERIMENT_NAME_SQRA,
    "train_days":      TRAIN_DAYS_ROLLING_SQRA,
    "quantiles":       QUANTILES_SQRA,
    "test_start":      str(TEST_START_SQRA.date()),
    "test_end":        str(TEST_END_SQRA.date()),
    "import_paths":    [str(p) for p in IMPORT_PATHS_SQRA],
    "created_at":      datetime.now().strftime("%Y-%m-%d %H:%M"),
}

with open(EXPORT_BASE_SQRA / "config.json", "w") as f:
    json.dump(meta, f, indent=2)

print(f"✓ Saved  →  {EXPORT_BASE_SQRA}")

#### Quick Evaluation: Plot and Metrics

In [ ]:
# Plot
plot_prob_forecast(
    df_forecast=res_sqra_sorted,
    quantile_cols=QUANTILE_COLS,
    start_date=str(TEST_START_SQRA.date()),
    end_date=str(TEST_END_SQRA.date()),
    model_info=f"SQRA | d{TRAIN_DAYS_ROLLING_SQRA}",
)

# Metrics & save
df_results = evaluate_probabilistic_forecasts(
    df=res_sqra_sorted,
    start_date=str(TEST_START_SQRA.date()),
    end_date=str(TEST_END_SQRA.date()),
    quantiles=QUANTILES_SQRA,
    y_true_col="y_true",
)
df_results.to_csv(EXPORT_BASE_SQRA / "metrics.csv", index=False)

print(f"✓ Metrics  →  {EXPORT_BASE_SQRA}")

df_results